# Intercase summary comparison table

Reads the four `summary.csv` files below and reshapes them into one table.

| Source file | Produced by |
|---|---|
| `results/inter_case_camargo_synthetic/summary.csv` | `intercase_camargo_suffix.ipynb` (synthetic) |
| `results/inter_case_camargo_real/summary.csv` | `intercase_camargo_suffix.ipynb` (real-life/`ssd`) |
| `results/inter_case_bukhsh_synthetic/summary.csv` | `intercase_bukhsh_remaining_time.ipynb` (synthetic) |
| `results/inter_case_bukhsh_real/summary.csv` | `intercase_bukhsh_remaining_time.ipynb` (real-life/`ssd`) |

The two real-life files are optional -- missing either just drops that model's
`is_real=True` rows.

Output: `results/inter_case_summary_comparison.csv`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent.parent
RESULTS = ROOT / "results"

print(f"ROOT    = {ROOT}")
print(f"RESULTS = {RESULTS}")

ROOT    = /Users/mi98gr/Documents/PhD/27_coding_local/06_System_level_prediction
RESULTS = /Users/mi98gr/Documents/PhD/27_coding_local/06_System_level_prediction/results


## Build and save

In [2]:
def build_intercase_comparison_csv():
    camargo_summary = pd.read_csv(RESULTS / 'inter_case_camargo_synthetic' / 'summary.csv')
    bukhsh_summary = pd.read_csv(RESULTS / 'inter_case_bukhsh_synthetic' / 'summary.csv')
    camargo_real_path = RESULTS / 'inter_case_camargo_real' / 'summary.csv'
    bukhsh_real_path = RESULTS / 'inter_case_bukhsh_real' / 'summary.csv'
    camargo_real_summary = pd.read_csv(camargo_real_path) if camargo_real_path.exists() else None
    bukhsh_real_summary = pd.read_csv(bukhsh_real_path) if bukhsh_real_path.exists() else None

    regime_map = [('_plain', 'plain'), ('', 'first'), ('_half', 'half')]
    sources = [('camargo', camargo_summary, False), ('bukhsh', bukhsh_summary, False)]
    if camargo_real_summary is not None:
        sources.append(('camargo', camargo_real_summary, True))
    if bukhsh_real_summary is not None:
        sources.append(('bukhsh', bukhsh_real_summary, True))

    rows = []
    for model_name, df, is_real in sources:
        d = df.set_index('dataset')
        for ds in d.index:
            for suffix, regime_label in regime_map:
                cc_i_col, cc_b_col, cc_p_col = f'cc_mae__intercase{suffix}', f'cc_mae__baseline{suffix}', f'cc_mae_pct_change{suffix}'
                tt_i_col, tt_b_col, tt_p_col = f'tt_mae__intercase{suffix}', f'tt_mae__baseline{suffix}', f'tt_mae_pct_change{suffix}'
                if cc_p_col not in d.columns:
                    continue
                rows.append(dict(
                    model=model_name, is_real=is_real, dataset=ds, regime=regime_label,
                    cc_mae_intercase=d.loc[ds, cc_i_col] if cc_i_col in d.columns else None,
                    cc_mae_baseline=d.loc[ds, cc_b_col] if cc_b_col in d.columns else None,
                    cc_mae_pct_change=d.loc[ds, cc_p_col],
                    tt_mae_intercase=d.loc[ds, tt_i_col] if tt_i_col in d.columns else None,
                    tt_mae_baseline=d.loc[ds, tt_b_col] if tt_b_col in d.columns else None,
                    tt_mae_pct_change=d.loc[ds, tt_p_col] if tt_p_col in d.columns else None,
                ))
    return pd.DataFrame(rows)


intercase_comparison_df = build_intercase_comparison_csv()
out_csv = RESULTS / 'inter_case_summary_comparison.csv'
intercase_comparison_df.to_csv(out_csv, index=False)
print(f'saved -> {out_csv}  ({len(intercase_comparison_df)} rows)')
intercase_comparison_df

saved -> /Users/mi98gr/Documents/PhD/27_coding_local/06_System_level_prediction/results/inter_case_summary_comparison.csv  (108 rows)


,model,is_real,dataset,regime,cc_mae_intercase,cc_mae_baseline,cc_mae_pct_change,tt_mae_intercase,tt_mae_baseline,tt_mae_pct_change
0,camargo,False,loan_combined,plain,775.622222,576.755556,34.48,442.708110,337.455989,31.19
1,camargo,False,loan_combined,first,833.977778,622.133333,34.05,481.877755,325.872015,47.87
2,camargo,False,loan_combined,half,457.800000,365.911111,25.11,236.243825,187.511678,25.99
3,camargo,False,loan_drift,plain,141.488889,172.800000,-18.12,135.781696,161.391113,-15.87
4,camargo,False,loan_drift,first,150.644444,170.377778,-11.58,133.904774,149.339006,-10.34
...,...,...,...,...,...,...,...,...,...,...
103,bukhsh,True,helpdesk,first,6.391000,6.554000,-2.49,142.254000,125.765000,13.11
104,bukhsh,True,helpdesk,half,6.692000,4.637000,44.32,141.341000,141.025000,0.22
105,bukhsh,True,sepsis,plain,34.990000,35.312000,-0.91,1040.366000,1032.421000,0.77
106,bukhsh,True,sepsis,first,39.635000,40.438000,-1.99,1029.692000,1025.466000,0.41
